# Lab 1.6 — Retrieve Wikipedia article examples with ChromaDB

This notebook adds retrieval-augmented generation to the Lab 1.5 agent. ChromaDB indexes 1,000 labeled synthetic article pairs. For a new article, the notebook retrieves three neighbors, builds an augmented Gemini prompt, and shows the resulting structure.

**Flow** · X → embed the title and article body → find three similar inputs in Chroma → add their labeled pairs to the prompt → call Gemini → validate and display y.

## Setup

Add `GEMINI_KEY` under Colab **Secrets** and grant notebook access. The setup cell uses a local checkout when available. Before publishing the lab files to GitHub, upload `iis-colab-input.zip` to `/content` using Colab’s **Files** panel. The ZIP should contain `article_agent.py`, `data/`, and `prompts/` at its root.

In [ ]:
%pip -q install google-genai==2.12.1 google-auth==2.49.0 chromadb==1.5.9 opentelemetry-api==1.42.1 opentelemetry-sdk==1.42.1 opentelemetry-exporter-otlp-proto-grpc==1.42.1 "oauthlib>=3.2,<4"

In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

ROOT = Path("/content/IIS-nd")
if Path("article_agent.py").is_file():
    ROOT = Path.cwd()
elif Path("../article_agent.py").is_file():
    ROOT = Path.cwd().parent
elif Path("/content/iis-colab-input.zip").is_file() and not (ROOT / "article_agent.py").is_file():
    with ZipFile("/content/iis-colab-input.zip") as archive:
        archive.extractall(ROOT)
elif not ROOT.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/MIM323/IIS-nd.git", str(ROOT)], check=True
    )
if not (ROOT / "article_agent.py").is_file():
    raise FileNotFoundError("Point ROOT to the extracted repository containing article_agent.py.")
sys.path.insert(0, str(ROOT))

from article_agent import MODEL, build_prompt, prepare, preview, read_jsonl, run_agent

instructions = (ROOT / "prompts" / "instructions.txt").read_text(encoding="utf-8")
fixed_examples = read_jsonl(ROOT / "data" / "examples.jsonl")
holdout = read_jsonl(ROOT / "data" / "holdout.jsonl")
target = holdout[0]["X"]
MODEL_ID = MODEL  # Change this if your project uses a different Gemini model.
print("Gemini model:", MODEL_ID)

## Index the examples

Chroma's default embedding function uses all-MiniLM-L6-v2 and can download its weights on the first run. The index uses cosine distance. It embeds the example title and body and keeps the full labeled pair as metadata. The collection is persisted in the current Colab session and keyed by the corpus content so the cell can be rerun.

In [ ]:
from article_agent import index_examples, retrieve_three, score_corpus

corpus = read_jsonl(ROOT / "data" / "retrieval_examples.jsonl")
assert len(corpus) >= 1_000
assert len({row["id"] for row in corpus}) == len(corpus)
assert {row["X"]["title"] for row in holdout}.isdisjoint(row["X"]["title"] for row in corpus)
collection = index_examples(Path.cwd() / ".chroma", corpus)
print("Indexed examples:", collection.count())

## Show X, three retrieved pairs, and the augmented prompt

The rankings reflect similarity to the target's title and body. Lower cosine distance means a closer match within this index. Example facts illustrate the format. The prompt limits evidence in the answer to the target text.

In [ ]:
def show_json(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))

print("Input X")
show_json(target)
neighbors = retrieve_three(collection, target)
print("\nThree ranked examples, with their full X and y")
show_json(neighbors)
print("\nAugmented prompt")
print(build_prompt(instructions, target, neighbors))

## Generate and validate y

Gemini performs classification, entity recognition, information extraction and section assignment. The validator checks the response shape, template, literal evidence, and paragraph coverage before the notebook displays a preview.

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_KEY"))

In [ ]:
rag_output = run_agent(client, instructions, target, neighbors, model=MODEL_ID)
print("Generated y")
show_json(rag_output)
print("\nReviewed article layout")
print(preview(target, rag_output))

## Compare with the fixed few-shot baseline

Both conditions use the same model, instructions, three demonstrations per input, and six held-out cases. The baseline uses the fixed Lab 1.2 examples. RAG chooses its examples by input similarity. Report category accuracy, valid output rate and micro-averaged precision, recall and F1 for field values, entity spans and paragraph headings. Invalid model outputs count as failed predictions. API errors surface so that quota and connectivity problems stay visible. Running the comparison makes up to 12 additional calls, with the RAG demonstration reused when available.

In [ ]:
modes = ("fixed_few_shot", "rag")
predictions = {mode: [] for mode in modes}
details = []
for sample in holdout:
    observation = sample["X"]
    retrieved = neighbors if observation == target else retrieve_three(collection, observation)
    report = {
        "id": sample["id"],
        "retrieved": [
            {"id": row["id"], "distance": row["distance"], "category": row["y"]["category"]}
            for row in retrieved
        ],
    }
    for mode, examples in zip(modes, (fixed_examples, retrieved), strict=True):
        try:
            predicted = globals().get("rag_output") if mode == "rag" and observation == target else None
            if predicted is None:
                predicted = run_agent(client, instructions, observation, examples, model=MODEL_ID)
            report[mode] = {"y": predicted}
        except ValueError as error:
            predicted = None
            report[mode] = {"error": str(error)}
        predictions[mode].append(predicted)
    details.append(report)

gold = [row["y"] for row in holdout]
scores = {mode: score_corpus(gold, predictions[mode]) for mode in modes}
deltas = {key: scores["rag"][key] - score for key, score in scores["fixed_few_shot"].items()}
for key, delta in deltas.items():
    print(f"{key:20} fixed={scores['fixed_few_shot'][key]:.3f} "
          f"rag={scores['rag'][key]:.3f} delta={delta:+.3f}")
result = {"model": MODEL_ID, "n": len(holdout), "scores": scores, "rag_delta": deltas, "cases": details}
Path("rag-results.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")

## Reflection

RAG supplies examples that can illustrate which infobox fields to fill or omit for a particular article. In a local Chroma run, all three neighbors matched the subject category for each of the four biography and country cases. The two other cases received neighbors from different classes. The non-sovereign island attracted country examples, while the publishing house attracted biography examples. These results show both where topical retrieval gives useful context and where it can select misleading examples. The paired Gemini comparison above measures the effect on output quality while keeping the number of examples constant. A live run is needed to establish whether the changes in context improve predictions, and a larger benchmark of real articles would give a better estimate of reliability.